In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/leash-BELKA/sample_submission.csv
/kaggle/input/competitions/leash-BELKA/train.parquet
/kaggle/input/competitions/leash-BELKA/test.parquet
/kaggle/input/competitions/leash-BELKA/train.csv
/kaggle/input/competitions/leash-BELKA/test.csv
/kaggle/input/datasets/t8101349/submission/submission.csv
/kaggle/input/datasets/t8101349/train-data1/optimized_train_data.parquet
/kaggle/input/notebooks/t8101349/predict-new-medicines-with-belka-merge-data/optimized_train_data.parquet
/kaggle/input/notebooks/t8101349/predict-new-medicines-with-belka-merge-data/__results__.html
/kaggle/input/notebooks/t8101349/predict-new-medicines-with-belka-merge-data/__notebook__.ipynb
/kaggle/input/notebooks/t8101349/predict-new-medicines-with-belka-merge-data/__output__.json
/kaggle/input/notebooks/t8101349/predict-new-medicines-with-belka-merge-data/final_merged_train_data.parquet
/kaggle/input/notebooks/t8101349/predict-new-medicines-with-belka-merge-data/custom.css
/kaggle/input/notebo

In [2]:
!pip install transformers torch rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 55.6 MB/s eta 0:00:00


In [3]:
import os
import gc
import torch
import polars as pl
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast
from transformers import DataCollatorWithPadding
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
import pyarrow.parquet as pq

In [4]:
'''
import os
import gc
import torch
import polars as pl
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast
from transformers import DataCollatorWithPadding
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
import pyarrow.parquet as pq

# --- 1. 配置與蛋白質預快取 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MOL_MODEL_NAME = "DeepChem/ChemBERTa-77M-MTR"
PROT_MODEL_NAME = "facebook/esm2_t12_35M_UR50D"

# 預先載入 Tokenizer
mol_tokenizer = AutoTokenizer.from_pretrained(MOL_MODEL_NAME)
prot_tokenizer = AutoTokenizer.from_pretrained(PROT_MODEL_NAME)
prot_model = AutoModel.from_pretrained(PROT_MODEL_NAME).to(device).eval()

def precompute_prot_embeddings():
    """預先計算 BELKA 的三種蛋白質 Embedding"""
    # 這裡建議填入三種蛋白質的完整 FASTA 序列，以下為示意代碼
    prot_sequences = {
        'BRD4': "MSAESGPGTRLRNLPVMGDGLETSQMSTTQAQAQPQPANAASTNPPPPETSNPNKPKRQTNQLQYLLRVVLKTLWKHQFAWPFQQPVDAVKLNLPDYYKIIKTPMDMGTIKKRLENNYYWNAQECIQDFNTMFTNCYIYNKPGDDIVLMAEALEKLFLQKINELPTEETEIMIVQAKGRGRGRKETGTAKPGVSTVPNTTQASTPPQTQTPQPNPPPVQATPHPFPAVTPDLIVQTPVMTVVPPQPLQTPPPVPPQPQPPPAPAPQPVQSHPPIIAATPQPVKTKKGVKRKADTTTPTTIDPIHEPPSLPPEPKTTKLGQRRESSRPVKPPKKDVPDSQQHPAPEKSSKVSEQLKCCSGILKEMFAKKHAAYAWPFYKPVDVEALGLHDYCDIIKHPMDMSTIKSKLEAREYRDAQEFGADVRLMFSNCYKYNPPDHEVVAMARKLQDVFEMRFAKMPDEPEEPVVAVSSPAVPPPTKVVAPPSSSDSSSDSSSDSDSSTDDSEEERAQRLAELQEQLKAVHEQLAALSQPQQNKPKKKKKKKKKKKK", 
        'HSA': "MKWVTFISLLFLFSSAYSRGVFRRDAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESAENCDKSLHTLFGDKLCTVATLRETYGEMADCCAKQEPERNECFLQHKDDNPNLPRLVRPEVDVMCTAFHDNEETFLKKYLYEIARRHPYFYAPELLFFAKRYKAAFTECCQAADKAACLLPKLDELRDEGKASSAKQRLKCASLQKFGERAFKAWAVARLSQRFPKAEFAEVSKLVTDLTKVHTECCHGDLLECADDRADLAKYICENQDSISSKLKECCEKPLLEKSHCIAEVENDEMPADLPSLAADFVESKDVCKNYAEAKDVFLGMFLYEYARRHPDYSVVLLLRLAKTYETTLEKCCAAADPHECYAKVFDEFKPLVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVEVSRNLGKVGSKCCKHPEAKRMPCAEDYLSVVLNQLCVLHEKTPVSDRVTKCCTESLVNRRPCFSALEVDETYVPKEFNAETFTFHADICTLSEKERQIKKQTALVELVKHKPKATKEQLKAVMDDFAAFVEKCCKADDKETCFAEEGKKLVAASQAALGL",
        'sEH': "MTLRAAVFDLDGVLALPAVFGVLGRTEEALALPRGLLNDAFQKGGPEGATTRLMKGEITLSQWIPLMEENCRKCSETAKVCLPKNFSIKEIFDKAISARKINRPMLQAALMLRKKGFTTAILTNTWLDDRAERDGLAQLMCELKMHFDFLIESCQVGMVKPEPQIYKFLLDTLKASPSEVVFLDDIGANLKPARDLGMVTILVQDTDTALKELEKVTGIQLLNTPAPLPTSCNPSDMSHGYVTVKPRVRLHFVELGSGPAVCLCHGFPESWYSWRYQIPALAQAGYRVLAMDMKGYGESSAPPEIEEYCMEVLCKEMVTFLDKLGLSQAVFIGHDWGGMLVWYMALFYPERVRAVASLNTPFIPANPNMSPLESIKANPVFDYQLYFQEPGVAEAELEQNLSRTFKSLFRASDESVLSMHKVCEAGGLFVNSPEEPSLSRMVTEEEIQFYVQQFKKSGFRGPLNWYRNMERNWKWACKSLGRKILIPALMVTAEKDFVLVPQMSQHMEDWIPHLKRGHIEDCGHWTQMDKPTEVNQILIKWLDSDARNPPVVSKM"
    }
    prot_map = {}
    for name, seq in prot_sequences.items():
        inputs = prot_tokenizer(seq, return_tensors="pt", truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            outputs = prot_model(**inputs)
            # 取 Mean Pooling
            prot_map[name] = outputs.last_hidden_state.mean(dim=1).cpu().numpy().squeeze()
    return prot_map

# 這是我們預處理好的 map
prot_map = precompute_prot_embeddings()

class BELKAPreciseModel(nn.Module):
    # 將預設值設為 0，第一階段完全凍結
    def __init__(self, hidden_dim=256, prot_dim=480, unfreeze_last_n_layers=0):
        super().__init__()
        
        print("📥 正在載入 ChemBERTa-77M-MTR 預訓練權重...")
        self.chemberta = AutoModel.from_pretrained("DeepChem/ChemBERTa-77M-MTR")
        
        # 預設：全面凍結
        for param in self.chemberta.parameters():
            param.requires_grad = False
            
        # 依據參數動態解凍
        if unfreeze_last_n_layers > 0:
            print(f"🔓 正在解凍 ChemBERTa 的最後 {unfreeze_last_n_layers} 層 Encoder...")
            for layer in self.chemberta.encoder.layer[-unfreeze_last_n_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
                    
        self.mol_proj = nn.Sequential(
            nn.Linear(384, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        self.prot_fc = nn.Sequential(
            nn.Linear(prot_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, 
            nhead=4, 
            batch_first=True, 
            dropout=0.1,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, mol_ids, mol_mask, prot_emb):
        """
        mol_ids: [Batch, Seq_Len]
        mol_mask: [Batch, Seq_Len]
        prot_emb: [Batch, 480]
        """
        # 判斷 ChemBERTa 是否有任何參數需要梯度
        # (如果 unfreeze_last_n_layers = 0，這會是 False；如果是 2，這會是 True)
        needs_grad = any(p.requires_grad for p in self.chemberta.parameters())
        
        # 動態控制梯度引擎：只有在需要的時候才開啟追蹤
        with torch.set_grad_enabled(needs_grad):
            chem_outputs = self.chemberta(input_ids=mol_ids, attention_mask=mol_mask)
        
        x_mol = chem_outputs.last_hidden_state  
        
        # 確保 x_mol 進入後方的 Transformer 前，具備正確的梯度屬性
        # 如果前面是 no_grad 算出來的，它預設不會有梯度要求，我們要手動確保它能傳遞梯度給後面的網路
        if not needs_grad:
            x_mol.requires_grad_(True)
            
        x_mol = self.mol_proj(x_mol)            # [Batch, Seq_Len, 256]
        x_prot = self.prot_fc(prot_emb).unsqueeze(1)      # [B, 1, 256]
        
        combined = torch.cat([x_prot, x_mol], dim=1)   # [B, Seq_Len + 1, 256]
        attn_out = self.transformer(combined)
        pooled = attn_out[:, 0, :] 
        
        return self.classifier(pooled)

class BELKATestDataset(Dataset):
    def __init__(self, df, prot_map):
        self.ids = df['id'].to_numpy()
        self.smiles = df['molecule_smiles'].to_numpy()
        self.prot_names = df['protein_name'].to_numpy()
        self.prot_map = prot_map

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        prot_vector = self.prot_map[self.prot_names[idx]]
        if isinstance(prot_vector, torch.Tensor):
            prot_vector = prot_vector.clone().detach().float()
        else:
            prot_vector = torch.tensor(prot_vector, dtype=torch.float32)

        return {
            "id": self.ids[idx],
            "smiles": self.smiles[idx], 
            "prot_emb": prot_vector
        }

# ==========================================
# 動態 Tokenize 的 Collate Function
# ==========================================
def create_test_collate_fn(tokenizer):
    def collate_fn(batch):
        ids = [item['id'] for item in batch]
        smiles_list = [item['smiles'] for item in batch]
        prot_embs = torch.stack([item['prot_emb'] for item in batch])

        # 在這裡做動態 Padding，節省大量 GPU 資源！
        tokenized = tokenizer(
            smiles_list,
            padding=True,
            truncation=True,
            max_length=128, # 取決於你 ChemBERTa 的設定
            return_tensors="pt"
        )

        return {
            "ids": ids,
            "mol_ids": tokenized["input_ids"],
            "mol_mask": tokenized["attention_mask"],
            "prot_emb": prot_embs
        }
    return collate_fn

# ==========================================
# generate_submission
# ==========================================
def generate_submission_optimized(model, test_file, mol_tokenizer, prot_map, device, output_name="submission.csv"):
    model.eval()
    
    print("使用 Polars 讀取測試集中...")
    test_df = pl.read_parquet(test_file, columns=["id", "molecule_smiles", "protein_name"])
    
    print("準備 DataLoader...")
    test_dataset = BELKATestDataset(test_df, prot_map)
    test_collate = create_test_collate_fn(mol_tokenizer)
    
    # 測試集的 Batch Size 可以開很大，因為不需要計算梯度 (Backprop)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=4096, 
        shuffle=False, # 不洗牌！
        num_workers=4, 
        pin_memory=True,
        collate_fn=test_collate
    )
    
    all_ids = []
    all_preds = []
    
    print("開始進行推論 (Inference)...")
    with torch.no_grad():
        # 使用 torch.amp.autocast 加速推論
        for batch in tqdm(test_loader, desc="Predicting"):
            batch_ids = batch["ids"]
            mol_ids = batch["mol_ids"].to(device)
            mol_mask = batch["mol_mask"].to(device)
            prot_emb = batch["prot_emb"].to(device)
            
            with torch.amp.autocast('cuda'):
                outputs = model(mol_ids, mol_mask, prot_emb)
            
            # 將 logits 轉成 0~1 的機率
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            
            all_ids.extend(batch_ids)
            all_preds.extend(probs)
            
    print(f"正在寫入 {output_name}...")
    # 使用 Pandas 快速寫出 CSV
    sub_df = pd.DataFrame({
        "id": all_ids,
        "binds": all_preds
    })
    sub_df.to_csv(output_name, index=False)
    print("✅ 提交檔案生成完畢！祝你 Leaderboard 暴衝！")

    del test_df, all_ids, all_preds, sub_df
    gc.collect()

# ==========================================
# 執行區塊
# ==========================================

model = BELKAPreciseModel(hidden_dim=256, prot_dim=480, unfreeze_last_n_layers=0).to(device)
model.load_state_dict(torch.load("/kaggle/input/models/t8101349/chemberta1v2/pytorch/default/1/best_model_chemberta_1_1.pth", map_location=device))

TEST_FILE = "/kaggle/input/competitions/leash-BELKA/test.parquet" 
SAMPLE_SUB_FILE = "/kaggle/input/competitions/leash-BELKA/sample_submission.csv"

if os.path.exists(TEST_FILE):
    generate_submission_optimized(model, TEST_FILE, mol_tokenizer, prot_map, device)
else:
    print(f"❌ 找不到測試檔: {TEST_FILE}，請檢查路徑。")

# 最後檢查行數
if os.path.exists("submission.csv") and os.path.exists(SAMPLE_SUB_FILE):
    # 用 Polars 掃描行數最快，不吃記憶體
    sub_count = pl.scan_csv("submission.csv").select(pl.len()).collect().item()
    sample_count = pl.scan_csv(SAMPLE_SUB_FILE).select(pl.len()).collect().item()
    
    if sub_count == sample_count:
        print(f"✅ 行數檢查完美通過: {sub_count} 筆數據")
    else:
        print(f"❌ 警告：行數不符！提交檔: {sub_count}, 範本: {sample_count}")
'''

'\nimport os\nimport gc\nimport torch\nimport polars as pl\nimport pandas as pd\nfrom tqdm import tqdm\nfrom torch.utils.data import DataLoader, Dataset\nfrom torch.amp import autocast\nfrom transformers import DataCollatorWithPadding\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport numpy as np\nfrom tqdm import tqdm\nfrom transformers import AutoTokenizer, AutoModel\nfrom sklearn.metrics import average_precision_score\nfrom sklearn.model_selection import train_test_split\nimport pyarrow.parquet as pq\n\n# --- 1. 配置與蛋白質預快取 ---\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nMOL_MODEL_NAME = "DeepChem/ChemBERTa-77M-MTR"\nPROT_MODEL_NAME = "facebook/esm2_t12_35M_UR50D"\n\n# 預先載入 Tokenizer\nmol_tokenizer = AutoTokenizer.from_pretrained(MOL_MODEL_NAME)\nprot_tokenizer = AutoTokenizer.from_pretrained(PROT_MODEL_NAME)\nprot_model = AutoModel.from_pretrained(PROT_MODEL_NAME).to(device).eval()\n\ndef precompute_prot_embeddings():\n    """預先計

In [5]:
'''
# --- 1. 數據優化抽樣 (Optimized Sampling) ---
# 參數設定
INPUT_FILE = "/kaggle/input/competitions/leash-BELKA/train.csv"
OUTPUT_FILE = "optimized_train_data.csv"
CHUNK_SIZE = 3_000_000

def csv_to_parquet_optimized(input_csv, output_parquet):
    print("正在將 CSV 轉換為高效能 Parquet 格式...")
    # 使用 Polars 讀取，並指定數據類型以節省空間
    df = pl.read_csv(input_csv, dtypes={
        "molecule_smiles": pl.Utf8,
        "protein_name": pl.Categorical,
        "binds": pl.Int8
    })
    df.write_parquet(output_parquet, compression="snappy")
    print(f"✅ 轉換完成: {output_parquet}")
    del df
    gc.collect()

def optimized_sampling_to_parquet(input_parquet, output_parquet, chunk_size=3_000_000, neg_to_pos_ratio=16):
    print("Sampling...")
    
    # 關鍵動作：建立 Parquet 檔案物件
    parquet_file = pq.ParquetFile(input_parquet)
    
    # ==========================================
    # 第一趟 (Pass 1)：母體普查
    # ==========================================
    print("\n [Pass 1] 正在進行全資料庫極速普查...")
    total_pos_count = 0
    total_neg_count = 0
    
    # 使用 iter_batches 來分批讀取，並轉成 Pandas DataFrame
    for batch in tqdm(parquet_file.iter_batches(batch_size=chunk_size, columns=['binds']), desc="Counting"):
        chunk = batch.to_pandas()
        total_pos_count += (chunk['binds'] == 1).sum()
        total_neg_count += (chunk['binds'] == 0).sum()
        
    print(f"母體總數 - 正樣本: {total_pos_count:,} | 負樣本: {total_neg_count:,}")
    
    # 精算抽樣率
    target_neg_count = total_pos_count * neg_to_pos_ratio
    exact_sampling_rate = target_neg_count / total_neg_count
    
    print(f" 目標負樣本數: {target_neg_count:,} (比例 15:1)")
    print(f" 算出抽樣機率: {exact_sampling_rate:.8f}")

    # ==========================================
    # 第二趟 (Pass 2)：精準資料提取
    # ==========================================
    print("\n [Pass 2] 正在進行精準資料提取與抽樣...")
    use_cols = ['molecule_smiles', 'protein_name', 'binds', 'buildingblock1_smiles']
    
    pos_chunks = []
    neg_chunks = []

    for batch in tqdm(parquet_file.iter_batches(batch_size=chunk_size, columns=use_cols), desc="Extracting & Sampling"):
        chunk = batch.to_pandas()
        
        # 1. 抓出所有正樣本
        pos_chunks.append(chunk[chunk['binds'] == 1])
        
        # 2. 抓出負樣本並使用機率抽樣
        neg_subset = chunk[chunk['binds'] == 0]
        neg_chunks.append(neg_subset.sample(frac=exact_sampling_rate, random_state=2))

    print("\n 讀取完畢！正在合併數據...")
    df_pos = pd.concat(pos_chunks)
    df_neg = pd.concat(neg_chunks)
    
    # 合併並洗牌
    df_final = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=2).reset_index(drop=True)

    print(f"最終樣本數: {len(df_final):,} (正: {len(df_pos):,}, 負: {len(df_neg):,})")
    
    # ==========================================
    # 寫入與輸出
    # ==========================================
    print("正在轉換為 Polars 並寫入高效能 Parquet...")
    
    pl_df = pl.from_pandas(df_final).with_columns([
        pl.col("molecule_smiles").cast(pl.Utf8),
        pl.col("buildingblock1_smiles").cast(pl.Utf8),  
        pl.col("protein_name").cast(pl.Categorical),    
        pl.col("binds").cast(pl.Int8)
    ])
    
    pl_df.write_parquet(output_parquet, compression="snappy")
    
    print(f"完美轉換！檔案已儲存至 {output_parquet}")
    
    del df_pos, df_neg, df_final, pl_df
    gc.collect()



optimized_sampling_to_parquet("/kaggle/input/competitions/leash-BELKA/train.parquet", "optimized_train_data.parquet")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
'''

'\n# --- 1. 數據優化抽樣 (Optimized Sampling) ---\n# 參數設定\nINPUT_FILE = "/kaggle/input/competitions/leash-BELKA/train.csv"\nOUTPUT_FILE = "optimized_train_data.csv"\nCHUNK_SIZE = 3_000_000\n\ndef csv_to_parquet_optimized(input_csv, output_parquet):\n    print("正在將 CSV 轉換為高效能 Parquet 格式...")\n    # 使用 Polars 讀取，並指定數據類型以節省空間\n    df = pl.read_csv(input_csv, dtypes={\n        "molecule_smiles": pl.Utf8,\n        "protein_name": pl.Categorical,\n        "binds": pl.Int8\n    })\n    df.write_parquet(output_parquet, compression="snappy")\n    print(f"✅ 轉換完成: {output_parquet}")\n    del df\n    gc.collect()\n\ndef optimized_sampling_to_parquet(input_parquet, output_parquet, chunk_size=3_000_000, neg_to_pos_ratio=16):\n    print("Sampling...")\n    \n    # 關鍵動作：建立 Parquet 檔案物件\n    parquet_file = pq.ParquetFile(input_parquet)\n    \n    # ==========================================\n    # 第一趟 (Pass 1)：母體普查\n    # ==========================================\n    print("\n [Pass 1] 正在進行全資料庫極速普查...")\n 

In [6]:
'''
import polars as pl
import gc

def merge_and_shuffle_data(main_parquet_path, external_csv_path, output_path, read_count_threshold=15):
    print("[Phase 1] 啟動外部資料融合...")
    
    # 1. 讀取主資料集
    print(f"讀取主資料集: {main_parquet_path}")
    df_main = pl.read_parquet(main_parquet_path)
    
    # 2. 讀取外部資料並進行嚴格篩選
    print(f"📥 讀取外部資料: {external_csv_path}")
    df_ext_raw = pl.read_csv(external_csv_path)
    
    # 過濾低頻雜訊 
    initial_count = len(df_ext_raw)
    df_ext_pos_filtered = df_ext_raw.filter(pl.col("read_count") >= read_count_threshold)
    print(f"pos_filtered -> {len(df_ext_pos_filtered)} 筆)")
    df_ext_neg_filtered = df_ext_raw.filter(pl.col("read_count") == 0)
    print(f"neg_filtered -> {len(df_ext_neg_filtered)} 筆)")
    
    # 保留 [Dy] 並使用 iso 立體結構
    df_positive = df_ext_pos_filtered.select([
        pl.col("new_structure").alias("molecule_smiles"),         # 完美保留 [Dy]
        pl.col("bb1_iso").alias("buildingblock1_smiles"),         # 完美保留立體化學特徵
        pl.lit("sEH").cast(pl.Categorical).alias("protein_name"), 
        pl.lit(1).cast(pl.Int8).alias("binds")                    # 因為經過高 read_count 篩選，現在可以安心給 1
    ])

    df_negative = df_ext_neg_filtered.select([
        pl.col("new_structure").alias("molecule_smiles"),         # 完美保留 [Dy]
        pl.col("bb1_iso").alias("buildingblock1_smiles"),         # 完美保留立體化學特徵
        pl.lit("sEH").cast(pl.Categorical).alias("protein_name"), 
        pl.lit(0).cast(pl.Int8).alias("binds")                    # 0
    ])

    df_ext = pl.concat([df_positive, df_negative])
    
    # 3. 合併與洗牌
    print("正在合併與全局洗牌...")
    df_main_subset = df_main.select(df_ext.columns) 
    df_combined = pl.concat([df_main_subset, df_ext])
    df_combined = df_combined.sample(fraction=1.0, shuffle=True, seed=42)

    # 5. 寫出 Parquet
    print(f"正在寫入最終訓練檔至: {output_path}")
    df_combined.write_parquet(output_path, compression="snappy")
    
    print(f"總樣本數：{len(df_combined):,} (新增 {len(df_ext_pos_filtered):,} 筆強效正樣本)(新增 {len(df_ext_neg_filtered):,} 筆嚴格負樣本)")
    
    del df_main, df_ext_raw, df_ext_pos_filtered, df_ext_neg_filtered, df_positive, df_negative, df_ext, df_combined, df_main_subset
    gc.collect()



# 執行合併 
merge_and_shuffle_data("optimized_train_data.parquet", "/kaggle/input/notebooks/chemdatafarmer/additional-seh-data/DNA_Labeled_Data.csv", "final_merged_train_data.parquet", read_count_threshold=15)
'''

'\nimport polars as pl\nimport gc\n\ndef merge_and_shuffle_data(main_parquet_path, external_csv_path, output_path, read_count_threshold=15):\n    print("[Phase 1] 啟動外部資料融合...")\n    \n    # 1. 讀取主資料集\n    print(f"讀取主資料集: {main_parquet_path}")\n    df_main = pl.read_parquet(main_parquet_path)\n    \n    # 2. 讀取外部資料並進行嚴格篩選\n    print(f"📥 讀取外部資料: {external_csv_path}")\n    df_ext_raw = pl.read_csv(external_csv_path)\n    \n    # 過濾低頻雜訊 \n    initial_count = len(df_ext_raw)\n    df_ext_pos_filtered = df_ext_raw.filter(pl.col("read_count") >= read_count_threshold)\n    print(f"pos_filtered -> {len(df_ext_pos_filtered)} 筆)")\n    df_ext_neg_filtered = df_ext_raw.filter(pl.col("read_count") == 0)\n    print(f"neg_filtered -> {len(df_ext_neg_filtered)} 筆)")\n    \n    # 保留 [Dy] 並使用 iso 立體結構\n    df_positive = df_ext_pos_filtered.select([\n        pl.col("new_structure").alias("molecule_smiles"),         # 完美保留 [Dy]\n        pl.col("bb1_iso").alias("buildingblock1_smiles"),         # 完美保留立體化學特徵

In [7]:
import polars as pl
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

def load_and_split_data(parquet_path, n_splits=5):
    print("正在透過 Polars 載入數據...")
    df = pl.read_parquet(parquet_path)
    
    print("啟動 StratifiedGroupKFold 嚴格切分 (BB1 隔離)...")
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    y = df['binds'].to_numpy()
    groups = df['buildingblock1_smiles'].to_numpy()
    X = np.zeros(len(y)) 
    
    for train_idx, val_idx in sgkf.split(X, y, groups):
        train_df = df[train_idx].to_pandas()
        val_df = df[val_idx].to_pandas()
        break 
        
    print(f"分割完成 | 訓練集: {len(train_df)} | 驗證集: {len(val_df)}")
    
    # 4. 嚴格洩漏檢查 (此時還需要用到 BB1)
    train_groups = set(train_df['buildingblock1_smiles'].unique())
    val_groups = set(val_df['buildingblock1_smiles'].unique())
    leakage = train_groups.intersection(val_groups)
    
    if len(leakage) == 0:
        print("洩漏檢查完美通過：訓練集與驗證集完全沒有重複的化學骨架 (BB1)！")
    else:
        print(f"警告：發現洩漏！重疊的 BB1 數量: {len(leakage)}")
    
    # ==========================================
    # 卸載 BB1 欄位
    # ==========================================
    print("🧹 正在清理暫存欄位，無縫對齊下游 Dataset...")
    train_df = train_df.drop(columns=['buildingblock1_smiles'])
    val_df = val_df.drop(columns=['buildingblock1_smiles'])
    
    return train_df, val_df


print("開始執行Scaffold Split 數據載入...")
train_df, val_df = load_and_split_data("/kaggle/input/notebooks/t8101349/predict-new-medicines-with-belka-merge-data/final_merged_train_data.parquet", n_splits=5) ###

import gc
gc.collect()

print(f"分割完成！訓練集大小: {len(train_df)}, 驗證集大小: {len(val_df)}")


開始執行Scaffold Split 數據載入...
正在透過 Polars 載入數據...
啟動 StratifiedGroupKFold 嚴格切分 (BB1 隔離)...
分割完成 | 訓練集: 20242712 | 驗證集: 8988828
洩漏檢查完美通過：訓練集與驗證集完全沒有重複的化學骨架 (BB1)！
🧹 正在清理暫存欄位，無縫對齊下游 Dataset...
分割完成！訓練集大小: 20242712, 驗證集大小: 8988828


In [8]:
import torchvision
import torch.nn as nn

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=3.0, label_smoothing=0.005):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        targets = targets.view(-1, 1).float()
        inputs = inputs.view(-1, 1).float()
        
        if self.label_smoothing > 0:
            smoothed_targets = targets * (1.0 - self.label_smoothing) + 0.5 * self.label_smoothing
        else:
            smoothed_targets = targets

        bce_loss = self.bce(inputs, smoothed_targets)
        probas = torch.sigmoid(inputs)
        p_t = probas * targets + (1 - probas) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        
        loss = alpha_t * focal_weight * bce_loss
        return loss.mean()

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import polars as pl
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import average_precision_score

# --- 1. 配置與蛋白質預快取 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MOL_MODEL_NAME = "DeepChem/ChemBERTa-77M-MTR"
PROT_MODEL_NAME = "facebook/esm2_t12_35M_UR50D"

# 預先載入 Tokenizer
mol_tokenizer = AutoTokenizer.from_pretrained(MOL_MODEL_NAME)
prot_tokenizer = AutoTokenizer.from_pretrained(PROT_MODEL_NAME)
prot_model = AutoModel.from_pretrained(PROT_MODEL_NAME).to(device).eval()

def precompute_prot_embeddings():
    """預先計算 BELKA 的三種蛋白質 Embedding"""
    prot_sequences = {
        'BRD4': "MSAESGPGTRLRNLPVMGDGLETSQMSTTQAQAQPQPANAASTNPPPPETSNPNKPKRQTNQLQYLLRVVLKTLWKHQFAWPFQQPVDAVKLNLPDYYKIIKTPMDMGTIKKRLENNYYWNAQECIQDFNTMFTNCYIYNKPGDDIVLMAEALEKLFLQKINELPTEETEIMIVQAKGRGRGRKETGTAKPGVSTVPNTTQASTPPQTQTPQPNPPPVQATPHPFPAVTPDLIVQTPVMTVVPPQPLQTPPPVPPQPQPPPAPAPQPVQSHPPIIAATPQPVKTKKGVKRKADTTTPTTIDPIHEPPSLPPEPKTTKLGQRRESSRPVKPPKKDVPDSQQHPAPEKSSKVSEQLKCCSGILKEMFAKKHAAYAWPFYKPVDVEALGLHDYCDIIKHPMDMSTIKSKLEAREYRDAQEFGADVRLMFSNCYKYNPPDHEVVAMARKLQDVFEMRFAKMPDEPEEPVVAVSSPAVPPPTKVVAPPSSSDSSSDSSSDSDSSTDDSEEERAQRLAELQEQLKAVHEQLAALSQPQQNKPKKKKKKKKKKKK", 
        'HSA': "MKWVTFISLLFLFSSAYSRGVFRRDAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESAENCDKSLHTLFGDKLCTVATLRETYGEMADCCAKQEPERNECFLQHKDDNPNLPRLVRPEVDVMCTAFHDNEETFLKKYLYEIARRHPYFYAPELLFFAKRYKAAFTECCQAADKAACLLPKLDELRDEGKASSAKQRLKCASLQKFGERAFKAWAVARLSQRFPKAEFAEVSKLVTDLTKVHTECCHGDLLECADDRADLAKYICENQDSISSKLKECCEKPLLEKSHCIAEVENDEMPADLPSLAADFVESKDVCKNYAEAKDVFLGMFLYEYARRHPDYSVVLLLRLAKTYETTLEKCCAAADPHECYAKVFDEFKPLVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVEVSRNLGKVGSKCCKHPEAKRMPCAEDYLSVVLNQLCVLHEKTPVSDRVTKCCTESLVNRRPCFSALEVDETYVPKEFNAETFTFHADICTLSEKERQIKKQTALVELVKHKPKATKEQLKAVMDDFAAFVEKCCKADDKETCFAEEGKKLVAASQAALGL",
        'sEH': "MTLRAAVFDLDGVLALPAVFGVLGRTEEALALPRGLLNDAFQKGGPEGATTRLMKGEITLSQWIPLMEENCRKCSETAKVCLPKNFSIKEIFDKAISARKINRPMLQAALMLRKKGFTTAILTNTWLDDRAERDGLAQLMCELKMHFDFLIESCQVGMVKPEPQIYKFLLDTLKASPSEVVFLDDIGANLKPARDLGMVTILVQDTDTALKELEKVTGIQLLNTPAPLPTSCNPSDMSHGYVTVKPRVRLHFVELGSGPAVCLCHGFPESWYSWRYQIPALAQAGYRVLAMDMKGYGESSAPPEIEEYCMEVLCKEMVTFLDKLGLSQAVFIGHDWGGMLVWYMALFYPERVRAVASLNTPFIPANPNMSPLESIKANPVFDYQLYFQEPGVAEAELEQNLSRTFKSLFRASDESVLSMHKVCEAGGLFVNSPEEPSLSRMVTEEEIQFYVQQFKKSGFRGPLNWYRNMERNWKWACKSLGRKILIPALMVTAEKDFVLVPQMSQHMEDWIPHLKRGHIEDCGHWTQMDKPTEVNQILIKWLDSDARNPPVVSKM"
    }
    prot_map = {}
    for name, seq in prot_sequences.items():
        inputs = prot_tokenizer(seq, return_tensors="pt", truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            outputs = prot_model(**inputs)
            # 取 Mean Pooling
            prot_map[name] = outputs.last_hidden_state.mean(dim=1).cpu().numpy().squeeze()
    return prot_map

# 預處理好的 map
prot_map = precompute_prot_embeddings()

# --- 2. 數據集與模型定義 ---

class BELKADynamicDataset(Dataset):
    def __init__(self, df, prot_map):
        self.smiles = df['molecule_smiles'].values
        self.prot_names = df['protein_name'].values
        self.labels = df['binds'].values
        self.prot_map = prot_map

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # 1. 抓取蛋白質特徵
        prot_vector = self.prot_map[self.prot_names[idx]]
        if isinstance(prot_vector, torch.Tensor):
            prot_vector = prot_vector.clone().detach().float()
        else:
            prot_vector = torch.tensor(prot_vector, dtype=torch.float32)

        return {
            "smiles": self.smiles[idx], 
            "prot_emb": prot_vector,
            "label": torch.tensor([self.labels[idx]], dtype=torch.float32)
        }

def create_collate_fn(tokenizer):
    def collate_fn(batch):
        # 1. 將這個 Batch 裡所有的 SMILES 字串收集成一個 List
        smiles_list = [item["smiles"] for item in batch]
        
        # 2. 將蛋白質特徵與標籤打包成 Tensor
        prot_embs = torch.stack([item["prot_emb"] for item in batch])
        labels = torch.stack([item["label"] for item in batch])
        
        # 3. 動態 Padding 
        # padding=True 會自動找出 smiles_list 中最長的長度，並以此為準進行 Pad
        # truncation=True 依然保留，用來防範萬一有超級長（超過模型極限）的異常分子
        mol_enc = tokenizer(
            smiles_list, 
            padding=True,       # 關鍵：自動適應 Batch 內最大長度
            truncation=True, 
            return_tensors="pt"
        )
        
        # 4. 回傳字典格式
        return {
            "mol_ids": mol_enc["input_ids"],
            "mol_mask": mol_enc["attention_mask"],
            "prot_emb": prot_embs,
            "label": labels
        }
    return collate_fn

class BELKAPreciseModel(nn.Module):
    # 將預設值設為 0，第一階段完全凍結
    def __init__(self, hidden_dim=256, prot_dim=480, unfreeze_last_n_layers=0):
        super().__init__()
        
        print("正在載入 ChemBERTa-77M-MTR 預訓練權重...")
        self.chemberta = AutoModel.from_pretrained("DeepChem/ChemBERTa-77M-MTR")
        
        # 預設：全面凍結
        for param in self.chemberta.parameters():
            param.requires_grad = False
            
        # 依據參數動態解凍
        if unfreeze_last_n_layers > 0:
            print(f"正在解凍 ChemBERTa 的最後 {unfreeze_last_n_layers} 層 Encoder...")
            for layer in self.chemberta.encoder.layer[-unfreeze_last_n_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
                    
        self.mol_proj = nn.Sequential(
            nn.Linear(384, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        self.prot_fc = nn.Sequential(
            nn.Linear(prot_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, 
            nhead=4, 
            batch_first=True, 
            dropout=0.1,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, mol_ids, mol_mask, prot_emb):
        """
        mol_ids: [Batch, Seq_Len]
        mol_mask: [Batch, Seq_Len]
        prot_emb: [Batch, 480]
        """
        # 判斷 ChemBERTa 是否有任何參數需要梯度
        # (如果 unfreeze_last_n_layers = 0，這會是 False；如果是 2，這會是 True)
        needs_grad = any(p.requires_grad for p in self.chemberta.parameters())
        
        # 動態控制梯度引擎：只有在需要的時候才開啟追蹤
        with torch.set_grad_enabled(needs_grad):
            chem_outputs = self.chemberta(input_ids=mol_ids, attention_mask=mol_mask)
        
        x_mol = chem_outputs.last_hidden_state  
        
        # 確保 x_mol 進入後方的 Transformer 前，具備正確的梯度屬性
        # 如果前面是 no_grad 算出來的，它預設不會有梯度要求，我們要手動確保它能傳遞梯度給後面的網路
        if not needs_grad:
            x_mol.requires_grad_(True)
            
        x_mol = self.mol_proj(x_mol)            # [Batch, Seq_Len, 256]
        x_prot = self.prot_fc(prot_emb).unsqueeze(1)      # [B, 1, 256]
        
        combined = torch.cat([x_prot, x_mol], dim=1)   # [B, Seq_Len + 1, 256]
        attn_out = self.transformer(combined)
        pooled = attn_out[:, 0, :] 
        
        return self.classifier(pooled)



config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
import time
import torch
from torch.amp import autocast, GradScaler
from sklearn.metrics import average_precision_score
from tqdm import tqdm

def train_and_validate_optimized(
    model, train_loader, val_loader, optimizer, criterion, device, 
    epochs=10, time_limit_hours=7.0, 
    max_lr=5e-4,                           
    save_name="best_belka_model.pth",
    scheduler_type='onecycle'  
):
    # --- 優化三：動態切換 Scheduler ---
    if scheduler_type == 'onecycle':
        # 有 Warmup 階段
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, 
            max_lr=max_lr, 
            steps_per_epoch=len(train_loader), 
            epochs=epochs, 
            pct_start=0.3, 
            anneal_strategy='cos' 
        )
        print("📈 使用 OneCycleLR Scheduler (包含 Warmup)")
    elif scheduler_type == 'cosine':
        # 無 Warmup，穩定遞減
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=epochs * len(train_loader), 
            eta_min=1e-7
        )
        print("📉 使用 CosineAnnealingLR Scheduler (適合極限微調)")

    scaler = GradScaler('cuda')
    best_ap = 0.0
    start_time = time.time()

    for epoch in range(epochs):
        # --- 訓練階段 ---
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        
        for batch in pbar:
            ids = batch["mol_ids"].to(device)
            mask = batch["mol_mask"].to(device)  
            prot = batch["prot_emb"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            
            with autocast('cuda'):
                outputs = model(ids, mask, prot) 
                labels = labels.view_as(outputs) 
                loss = criterion(outputs.float(), labels.float())
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            train_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{scheduler.get_last_lr()[0]:.6e}"})

        # --- 驗證階段 ---
        model.eval()
        all_preds, all_labels = [], []
        val_loss = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                ids = batch["mol_ids"].to(device)
                mask = batch["mol_mask"].to(device)
                prot = batch["prot_emb"].to(device)
                labels = batch["label"].to(device)

                with autocast('cuda'):
                    outputs = model(ids, mask, prot) 
                    labels = labels.view_as(outputs)
                    v_loss = criterion(outputs.float(), labels.float())

                val_loss += v_loss.item()
                probs = torch.sigmoid(outputs.float()) 
                all_preds.extend(probs.cpu().numpy()) 
                all_labels.extend(labels.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        current_ap = average_precision_score(all_labels, all_preds)

        print(f"\n📊 Epoch {epoch+1} Summary:")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | AP: {current_ap:.4f}")

        if current_ap > best_ap:
            best_ap = current_ap
            if isinstance(model, nn.DataParallel):
                torch.save(model.module.state_dict(), save_name)
            else:
                torch.save(model.state_dict(), save_name)
                
            print(f"新紀錄！最佳模型已儲存為: {save_name} (AP: {best_ap:.4f})")
        print("-" * 30)
        
        # 時間控制
        elapsed_hours = (time.time() - start_time) / 3600
        if elapsed_hours > time_limit_hours:
            print(f"接近系統時間限制 ({time_limit_hours} 小時)，強制停止訓練以保留心血！")
            break

    return best_ap

In [11]:
# ==========================================
# 動態解凍控制函數
# ==========================================
def unfreeze_chemberta_layers(model, num_layers_to_unfreeze):
    """
    精準控制要解凍 ChemBERTa 的最後幾層。
    注意：傳入的 model 必須是『純淨版』(未被 DataParallel 包裝前)
    """
    # 1. 先把整個 ChemBERTa 凍結
    for param in model.chemberta.parameters():
        param.requires_grad = False
        
    # 2. 如果指定大於 0，解凍最後的 N 層
    if num_layers_to_unfreeze > 0:
        for layer in model.chemberta.encoder.layer[-num_layers_to_unfreeze:]:
            for param in layer.parameters():
                param.requires_grad = True
    print(f"已解凍 ChemBERTa 最後 {num_layers_to_unfreeze} 層！")



In [12]:
'''
import os
import torch
import polars as pl
from sklearn.model_selection import train_test_split # 應該從 sklearn 導入
from torch.utils.data import DataLoader, Dataset      # torch 只負責 Data 加載


# 設定設備
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 3072

print("🛠️ Preparing DataLoaders...")
train_dataset = BELKADynamicDataset(train_df, prot_map)
val_dataset = BELKADynamicDataset(val_df, prot_map)

custom_collate = create_collate_fn(mol_tokenizer)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=4, pin_memory=True, drop_last=True,
    collate_fn=custom_collate
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=4, pin_memory=True,
    collate_fn=custom_collate
)


# ==========================================
# 🥇 第一階段：隨機初始化層的高速收斂 (ChemBERTa 全凍結)
# ==========================================
print("\n" + "="*40)
print("啟動第一階段：訓練全新互動層 (ChemBERTa 全凍結)")
print("策略: OneCycleLR, Max LR=5e-4")
print("="*40)

# 1. 建立純淨模型
print("初始化基礎模型...")
model = BELKAPreciseModel(hidden_dim=256, prot_dim=480).to(device)

# 2. 載入第一階段「熱身完畢」的最佳權重
print("載入 Phase 0 最佳權重...")
model.load_state_dict(torch.load("/kaggle/input/models/t8101349/chemberta1/pytorch/default/1/best_model_chemberta_1.pth", map_location=device))

unfreeze_chemberta_layers(model, num_layers_to_unfreeze=0)

if torch.cuda.device_count() > 1:
    print(f"偵測到 {torch.cuda.device_count()} 張 GPU！啟動 DataParallel 雙卡平行運算！")
    model = nn.DataParallel(model)

criterion_phase1 = FocalLoss(alpha=0.25, gamma=2.0, label_smoothing=0.05)

# 只將 requires_grad=True 的參數放入 Optimizer 
optimizer_phase1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-2)

best_ap_phase1 = train_and_validate_optimized(
    model=model, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer_phase1, criterion=criterion_phase1, device=device,
    epochs=5, 
    time_limit_hours=10.0,
    max_lr=3e-4, 
    save_name="best_model_chemberta_1.pth", 
    scheduler_type='onecycle'  
)

# ==========================================
# 🥈 第二階段：特徵對齊緩衝期 (解凍 ChemBERTa 最後 2 層)
# ==========================================
print("\n" + "="*40)
print("啟動第二階段：特徵對齊緩衝期 (解凍 ChemBERTa 最後 2 層)") 
print("策略: OneCycleLR, Max LR=1e-3")
print("="*40)

# 1. 建立純淨模型
print("初始化基礎模型...")
model = BELKAPreciseModel(hidden_dim=256, prot_dim=480).to(device)

# 2. 載入第一階段「熱身完畢」的最佳權重
print("載入 Phase 1 最佳權重...")
model.load_state_dict(torch.load("best_model_chemberta_1.pth", map_location=device))

unfreeze_chemberta_layers(model, num_layers_to_unfreeze=2)

if torch.cuda.device_count() > 1:
    print(f"偵測到 {torch.cuda.device_count()} 張 GPU！啟動 DataParallel 雙卡平行運算！")
    model = nn.DataParallel(model)
criterion_phase2 = FocalLoss(alpha=0.25, gamma=2.0, label_smoothing=0.05)

# 重新宣告 Optimizer，抓取解凍後的新參數
optimizer_phase2 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=6e-5, weight_decay=1e-2)

best_ap_phase2 = train_and_validate_optimized(  
    model=model, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer_phase2, criterion=criterion_phase2, device=device,
    epochs=4, 
    time_limit_hours=5.0,
    max_lr=6e-5,  # 使用中等 LR 讓解凍的層適應新的任務
    save_name="best_model_chemberta_2.pth",
    scheduler_type='onecycle' 
)

# ==========================================
# 🥉 第三階段：極限微調與困難樣本挖掘
# ==========================================
print("\n" + "="*40)
print("啟動第三階段：困難樣本極限深掘 (Gamma=3.0)") 
print("策略: CosineAnnealingLR, Max LR=1e-5")
print("="*40)


# 1. 建立純淨模型
print("初始化基礎模型...")
model = BELKAPreciseModel(hidden_dim=256, prot_dim=480).to(device)

# 2. 載入第二階段最穩定的權重
print("載入 Phase 2 最佳權重...")
model.load_state_dict(torch.load("best_model_chemberta_2.pth", map_location=device))

unfreeze_chemberta_layers(model, num_layers_to_unfreeze=4)

if torch.cuda.device_count() > 1:
    print(f"偵測到 {torch.cuda.device_count()} 張 GPU！啟動 DataParallel 雙卡平行運算！")
    model = nn.DataParallel(model)
    
# 調整 Loss 參數抓困難樣本 (Gamma 從 2.0 提升到 3.0)
criterion_phase3 = FocalLoss(alpha=0.25, gamma=3.0, label_smoothing=0.005)

# 重新宣告 Optimizer
optimizer_phase3 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-6, weight_decay=1e-2)

best_ap_phase3 = train_and_validate_optimized( 
    model=model, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer_phase3, criterion=criterion_phase3, device=device,
    epochs=3, 
    time_limit_hours=3.0,
    max_lr=5e-6,             
    save_name="best_model_chemberta_3.pth",
    scheduler_type='cosine'    # 無 Warmup，穩定遞減
)

print("\n🎉 三階段訓練全部完成！")
print(f"🥇 第一階段 AP: {best_ap_phase1:.4f}")
print(f"🥈 第二階段 AP: {best_ap_phase2:.4f}")
print(f"🥉 第三階段 AP: {best_ap_phase3:.4f}")
print(f"📁 最終權重已安全儲存於: best_model_chemberta_3.pth")
'''

'\nimport os\nimport torch\nimport polars as pl\nfrom sklearn.model_selection import train_test_split # 應該從 sklearn 導入\nfrom torch.utils.data import DataLoader, Dataset      # torch 只負責 Data 加載\n\n\n# 設定設備\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nprint(f"Using device: {device}")\n\nBATCH_SIZE = 3072\n\nprint("🛠️ Preparing DataLoaders...")\ntrain_dataset = BELKADynamicDataset(train_df, prot_map)\nval_dataset = BELKADynamicDataset(val_df, prot_map)\n\ncustom_collate = create_collate_fn(mol_tokenizer)\n\ntrain_loader = DataLoader(\n    train_dataset, batch_size=BATCH_SIZE, shuffle=True, \n    num_workers=4, pin_memory=True, drop_last=True,\n    collate_fn=custom_collate\n)\nval_loader = DataLoader(\n    val_dataset, batch_size=BATCH_SIZE, shuffle=False, \n    num_workers=4, pin_memory=True,\n    collate_fn=custom_collate\n)\n\n\n# ==========================================\n# 🥇 第一階段：隨機初始化層的高速收斂 (ChemBERTa 全凍結)\n# ==========================================\np

In [13]:

import os
import torch
import polars as pl
from sklearn.model_selection import train_test_split 
from torch.utils.data import DataLoader, Dataset      


# 設定設備
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 3072

print("🛠️ Preparing DataLoaders...")
train_dataset = BELKADynamicDataset(train_df, prot_map)
val_dataset = BELKADynamicDataset(val_df, prot_map)

custom_collate = create_collate_fn(mol_tokenizer)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=4, pin_memory=True, drop_last=True,
    collate_fn=custom_collate
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=4, pin_memory=True,
    collate_fn=custom_collate
)


# ==========================================
# 🥇 第一階段：隨機初始化層的高速收斂 (ChemBERTa 全凍結)
# ==========================================
print("\n" + "="*40)
print("啟動第一階段：訓練全新互動層 (ChemBERTa 全凍結)")
print("策略: OneCycleLR, Max LR=5e-4")
print("="*40)

# 1. 建立純淨模型
print("初始化基礎模型...")
model = BELKAPreciseModel(hidden_dim=256, prot_dim=480).to(device)

# 2. 載入第一階段「熱身完畢」的最佳權重
print("載入 Phase 0 最佳權重...")
model.load_state_dict(torch.load("/kaggle/input/models/t8101349/chemberta1v2/pytorch/default/1/best_model_chemberta_1_1.pth", map_location=device))

unfreeze_chemberta_layers(model, num_layers_to_unfreeze=0)

if torch.cuda.device_count() > 1:
    print(f"偵測到 {torch.cuda.device_count()} 張 GPU！啟動 DataParallel 雙卡平行運算！")
    model = nn.DataParallel(model)

criterion_phase1 = FocalLoss(alpha=0.25, gamma=2.0, label_smoothing=0.05)

# 只將 requires_grad=True 的參數放入 Optimizer 
optimizer_phase1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-2)

best_ap_phase1 = train_and_validate_optimized(
    model=model, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer_phase1, criterion=criterion_phase1, device=device,
    epochs=5, 
    time_limit_hours=3.0,
    max_lr=2e-4, 
    save_name="best_model_chemberta_1.pth", 
    scheduler_type='onecycle'  
)


print(f"🥇 第一階段 AP: {best_ap_phase1:.4f}")


Using device: cuda
🛠️ Preparing DataLoaders...

啟動第一階段：訓練全新互動層 (ChemBERTa 全凍結)
策略: OneCycleLR, Max LR=5e-4
初始化基礎模型...
正在載入 ChemBERTa-77M-MTR 預訓練權重...


pytorch_model.bin:   0%|          | 0.00/14.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
norm_mean                       | UNEXPECTED | 
regression.dense.weight         | UNEXPECTED | 
regression.out_proj.weight      | UNEXPECTED | 
regression.out_proj.bias        | UNEXPECTED | 
norm_std                        | UNEXPECTED | 
regression.dense.bias           | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


載入 Phase 0 最佳權重...


...

已解凍 ChemBERTa 最後 0 層！
偵測到 2 張 GPU！啟動 DataParallel 雙卡平行運算！
📈 使用 OneCycleLR Scheduler (包含 Warmup)



Epoch 1/5 [Val]: 100%|██████████| 2927/2927 [17:04<00:00,  2.86it/s]



📊 Epoch 1 Summary:
Train Loss: 0.0069 | Val Loss: 0.0084 | AP: 0.7304
新紀錄！最佳模型已儲存為: best_model_chemberta_1.pth (AP: 0.7304)
------------------------------


Epoch 2/5 [Val]: 100%|██████████| 2927/2927 [17:05<00:00,  2.86it/s]



📊 Epoch 2 Summary:
Train Loss: 0.0069 | Val Loss: 0.0084 | AP: 0.7273
------------------------------
接近系統時間限制 (3.0 小時)，強制停止訓練以保留心血！
🥇 第一階段 AP: 0.7304


In [14]:
import os
import gc
import torch
import polars as pl
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast
from transformers import DataCollatorWithPadding


class BELKATestDataset(Dataset):
    def __init__(self, df, prot_map):
        self.ids = df['id'].to_numpy()
        self.smiles = df['molecule_smiles'].to_numpy()
        self.prot_names = df['protein_name'].to_numpy()
        self.prot_map = prot_map

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        prot_vector = self.prot_map[self.prot_names[idx]]
        if isinstance(prot_vector, torch.Tensor):
            prot_vector = prot_vector.clone().detach().float()
        else:
            prot_vector = torch.tensor(prot_vector, dtype=torch.float32)

        return {
            "id": self.ids[idx],
            "smiles": self.smiles[idx], 
            "prot_emb": prot_vector
        }

# ==========================================
# 動態 Tokenize 的 Collate Function
# ==========================================
def create_test_collate_fn(tokenizer):
    def collate_fn(batch):
        ids = [item['id'] for item in batch]
        smiles_list = [item['smiles'] for item in batch]
        prot_embs = torch.stack([item['prot_emb'] for item in batch])

        # 在這裡做動態 Padding，節省大量 GPU 資源！
        tokenized = tokenizer(
            smiles_list,
            padding=True,
            truncation=True,
            max_length=128, # 取決於你 ChemBERTa 的設定
            return_tensors="pt"
        )

        return {
            "ids": ids,
            "mol_ids": tokenized["input_ids"],
            "mol_mask": tokenized["attention_mask"],
            "prot_emb": prot_embs
        }
    return collate_fn

# ==========================================
# generate_submission
# ==========================================
def generate_submission_optimized(model, test_file, mol_tokenizer, prot_map, device, output_name="submission.csv"):
    model.eval()
    
    print("使用 Polars 讀取測試集中...")
    test_df = pl.read_parquet(test_file, columns=["id", "molecule_smiles", "protein_name"])
    
    print("準備 DataLoader...")
    test_dataset = BELKATestDataset(test_df, prot_map)
    test_collate = create_test_collate_fn(mol_tokenizer)
    
    # 測試集的 Batch Size 可以開很大，因為不需要計算梯度 (Backprop)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=4096, 
        shuffle=False, # 不洗牌！
        num_workers=4, 
        pin_memory=True,
        collate_fn=test_collate
    )
    
    all_ids = []
    all_preds = []
    
    print("開始進行推論 (Inference)...")
    with torch.no_grad():
        # 使用 torch.amp.autocast 加速推論
        for batch in tqdm(test_loader, desc="Predicting"):
            batch_ids = batch["ids"]
            mol_ids = batch["mol_ids"].to(device)
            mol_mask = batch["mol_mask"].to(device)
            prot_emb = batch["prot_emb"].to(device)
            
            with torch.amp.autocast('cuda'):
                outputs = model(mol_ids, mol_mask, prot_emb)
            
            # 將 logits 轉成 0~1 的機率
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            
            all_ids.extend(batch_ids)
            all_preds.extend(probs)
            
    print(f"正在寫入 {output_name}...")
    # 使用 Pandas 快速寫出 CSV
    sub_df = pd.DataFrame({
        "id": all_ids,
        "binds": all_preds
    })
    sub_df.to_csv(output_name, index=False)
    print("✅ 提交檔案生成完畢！祝你 Leaderboard 暴衝！")

    del test_df, all_ids, all_preds, sub_df
    gc.collect()

# ==========================================
# 執行區塊
# ==========================================

model = BELKAPreciseModel(hidden_dim=256, prot_dim=480, unfreeze_last_n_layers=0).to(device)
model.load_state_dict(torch.load("best_model_chemberta_1.pth", map_location=device))

TEST_FILE = "/kaggle/input/competitions/leash-BELKA/test.parquet" 
SAMPLE_SUB_FILE = "/kaggle/input/competitions/leash-BELKA/sample_submission.csv"

if os.path.exists(TEST_FILE):
    generate_submission_optimized(model, TEST_FILE, mol_tokenizer, prot_map, device)
else:
    print(f"❌ 找不到測試檔: {TEST_FILE}，請檢查路徑。")

# 最後檢查行數
if os.path.exists("submission.csv") and os.path.exists(SAMPLE_SUB_FILE):
    # 用 Polars 掃描行數最快，不吃記憶體
    sub_count = pl.scan_csv("submission.csv").select(pl.len()).collect().item()
    sample_count = pl.scan_csv(SAMPLE_SUB_FILE).select(pl.len()).collect().item()
    
    if sub_count == sample_count:
        print(f"✅ 行數檢查完美通過: {sub_count} 筆數據")
    else:
        print(f"❌ 警告：行數不符！提交檔: {sub_count}, 範本: {sample_count}")

正在載入 ChemBERTa-77M-MTR 預訓練權重...


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
norm_mean                       | UNEXPECTED | 
regression.dense.weight         | UNEXPECTED | 
regression.out_proj.weight      | UNEXPECTED | 
regression.out_proj.bias        | UNEXPECTED | 
norm_std                        | UNEXPECTED | 
regression.dense.bias           | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


使用 Polars 讀取測試集中...
準備 DataLoader...
開始進行推論 (Inference)...


Predicting: 100%|██████████| 409/409 [05:58<00:00,  1.14it/s]


正在寫入 submission.csv...
✅ 提交檔案生成完畢！祝你 Leaderboard 暴衝！
✅ 行數檢查完美通過: 1674896 筆數據


In [15]:
import polars as pl
sub = pl.read_csv("submission.csv")
print("ID 是否嚴格遞增且無遺漏：", sub['id'].is_sorted())

# 把 submission 的結果跟 test.parquet 裡的 protein_name 對接起來看平均機率
test_df = pl.read_parquet("/kaggle/input/competitions/leash-BELKA/test.parquet", columns=["id", "protein_name"])
sub_merged = sub.join(test_df, on="id")
print(sub_merged.group_by("protein_name").agg(pl.col("binds").mean()))

ID 是否嚴格遞增且無遺漏： True
shape: (3, 2)
┌──────────────┬──────────┐
│ protein_name ┆ binds    │
│ ---          ┆ ---      │
│ str          ┆ f64      │
╞══════════════╪══════════╡
│ HSA          ┆ 0.078198 │
│ BRD4         ┆ 0.054757 │
│ sEH          ┆ 0.039445 │
└──────────────┴──────────┘
